# PLTreeSHAP + FastTreeSHAP

Runs **PLTreeSHAP + FastTreeSHAP** across all datasets and task types in the
`woodelfhd_depth_sweep_experiment`.
This notebook is **independent** and can run in parallel with the other method notebooks.

> ⚠️ **Python 3.8 required** — `fasttreeshap` and `pltreeshap` only support Python 3.8.
> This notebook uses a `micromamba` environment to run under Python 3.8 inside Colab.

### What this notebook does
1. Mounts Google Drive (results are saved there after each mission)
2. Copies and prepares the `run_python38_scripts.bash` helper from Drive
3. Clones `treebranchmarks` repo
4. Restores any previous partial results from Drive
5. Installs all dependencies into a Python 3.8 micromamba environment
6. Runs `woodelfhd_depth_sweep_experiment --method pltreeshap_fasttreeshap`
7. Writes partial results to Drive as `pltreeshap_fasttreeshap.json`

### Task routing
| Task | Library used |
|------|--------------|
| Background SHAP | `pltreeshap.PLTreeExplainer` |
| Background SHAP IV | `pltreeshap.PLTreeExplainer` |
| Path-Dependent SHAP | `fasttreeshap.TreeExplainer` (v2) |
| Path-Dependent SHAP IV | `fasttreeshap.TreeExplainer` (v1) |

### Depth limits (same as OriginalWoodelf)
The approach shares the same crash profile as OriginalWoodelf:
MEMORY_CRASH at high depths (D≥18 for SHAP, D≥15 for interactions on most datasets).

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ── Step 2: Configure paths ──────────────────────────────────────────────────
import pathlib

# Folder on Drive where results are persisted
DRIVE_FOLDER = pathlib.Path('/content/drive/MyDrive/ShapResearch/HighDepth/treebranchmark_experiments/results_jsons')
DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)

DRIVE_RESULT_PATH = DRIVE_FOLDER / 'pltreeshap_fasttreeshap.json'

# Path to the run_python38_scripts.bash helper on your Drive
BASH_SCRIPT_DRIVE_PATH = '/content/drive/MyDrive/ShapResearch/HighDepth/AAAI27/PLTreeSHAP/run_python38_scripts.bash'

print(f'Method cache will be saved to: {DRIVE_RESULT_PATH}')

Method cache will be saved to: /content/drive/MyDrive/ShapResearch/HighDepth/treebranchmark_experiments/results_jsons/pltreeshap_fasttreeshap.json


In [ ]:
# ── Step 3: Prepare the Python 3.8 bash helper ───────────────────────────────
import shutil
shutil.copy(BASH_SCRIPT_DRIVE_PATH, '/content/run_python38_scripts.bash')

# Fix Windows CRLF line endings if present
!sed -i 's/\r$//' /content/run_python38_scripts.bash
!chmod +x /content/run_python38_scripts.bash
print('Bash helper ready.')

Bash helper ready.


In [ ]:
# ── Step 4: Clone repositories and install woodelf ─────────────────────────
TREEBRANCHMARKS_URL = 'https://github.com/ron-wettenstein/TreeBranchMarks.git'

!git clone {TREEBRANCHMARKS_URL} /content/treebranchmarks

Cloning into '/content/treebranchmarks'...
remote: Enumerating objects: 502, done.
remote: Counting objects: 100% (502/502), done.
remote: Compressing objects: 100% (308/308), done.
remote: Total 502 (delta 332), reused 347 (delta 180), pack-reused 0 (from 0)
Receiving objects: 100% (502/502), 352.51 KiB | 4.30 MiB/s, done.
Resolving deltas: 100% (332/332), done.


In [ ]:
# ── Step 5: Restore method cache from a previous interrupted run ─────────────
import shutil, pathlib

cache_dir = pathlib.Path('/content/treebranchmarks/cache/method_results/woodelfhd_depth_sweep_experiment')
cache_dir.mkdir(parents=True, exist_ok=True)
local_cache_file = cache_dir / 'pltreeshap_fasttreeshap.json'

if DRIVE_RESULT_PATH.exists() and not local_cache_file.exists():
    shutil.copy(DRIVE_RESULT_PATH, local_cache_file)
    print(f'Restored method cache ({DRIVE_RESULT_PATH.stat().st_size // 1024} KB)')
else:
    print('No method cache to restore — starting fresh.')

Restored method cache (11 KB)


In [ ]:
!/content/run_python38_scripts.bash \
    --cwd /content/treebranchmarks \
    --pip "numpy==1.24.0 scipy pyarrow fastparquet pandas xgboost scikit-learn tqdm gdown" \
    --pip "Cython lightgbm" \
    --pip "woodelf_explainer" \
    --pip "-e /content/treebranchmarks" \
    --pip "fasttreeshap" \
    --pip "-v --no-build-isolation git+https://github.com/schufa-innovationlab/pltreeshap@main" \
    --run "-m benchmarks.woodelfhd_depth_sweep_experiment --method pltreeshap_fasttreeshap --result_location {DRIVE_RESULT_PATH}"


[py38-setup] cd /content/treebranchmarks

[py38-setup] Installing micromamba under /content/micromamba ...

[py38-setup] Creating env 'py38' with Python 3.8 ...
[+] 0.0s
[+] 0.1s
conda-forge/linux-64  ⣾  
conda-forge/noarch    ⣾  [+] 0.2s
conda-forge/linux-64   4%
conda-forge/noarch     6%[+] 0.3s
conda-forge/linux-64  17%
conda-forge/noarch    34%[+] 0.4s
conda-forge/linux-64  31%
conda-forge/noarch    62%[+] 0.5s
conda-forge/linux-64  45%
conda-forge/noarch    90%conda-forge/noarch                                
[+] 0.6s
conda-forge/linux-64  51%[+] 0.7s
conda-forge/linux-64  78%[+] 0.8s
conda-forge/linux-64 100%conda-forge/linux-64                              


Transaction

  Prefix: /content/micromamba/envs/py38

  Updating specs:

   - python=3.8
   - pip


  Package               Version  Build                 Channel          Size
──────────────────────────────────────────────────────────────────────────────
  Install:
────────────────────────────────────────────────────────

In [ ]:
# ── Step 7: Verify output ────────────────────────────────────────────────────
import json

with open(DRIVE_RESULT_PATH) as f:
    cache = json.load(f)

print(f'Entries in method cache: {len(cache)}')
if cache:
    sample = next(iter(cache.values()))
    print(f'Sample entry: {sample["_label"]}  →  {sample["running_time"]:.3f}s')
print(f'\nFile saved to: {DRIVE_RESULT_PATH}')

Entries in method cache: 41
Sample entry: Path-Dependent SHAP n=118108 m=0 D=6 T=100  →  43.158s

File saved to: /content/drive/MyDrive/ShapResearch/HighDepth/treebranchmark_experiments/results_jsons/pltreeshap_fasttreeshap.json


# Old Runs

In [ ]:
# ── Step 6: Run the experiment under Python 3.8 ───────────────────────────────
# fasttreeshap and pltreeshap require Python 3.8; micromamba provides the env.
#
# --method pltreeshap_fasttreeshap : only PLTreeSHAPFastTreeSHAPApproach is timed
# --result_location                : saves partial results to Drive after each mission
#
# pip groups (executed in order):
#   1. Core scientific stack pinned for Python 3.8 compatibility
#   2. Cython + LightGBM
#   3. woodelf_explainer (editable, from cloned repo)
#   4. treebranchmarks (editable, from cloned repo)
#   5. fasttreeshap
#   6. pltreeshap (from GitHub)

!/content/run_python38_scripts.bash \
    --cwd /content/treebranchmarks \
    --pip "numpy==1.24.0 scipy pyarrow fastparquet pandas xgboost scikit-learn tqdm gdown" \
    --pip "Cython lightgbm" \
    --pip "woodelf_explainer" \
    --pip "-e /content/treebranchmarks" \
    --pip "fasttreeshap" \
    --pip "-v --no-build-isolation git+https://github.com/schufa-innovationlab/pltreeshap@main" \
    --run "-m benchmarks.woodelfhd_depth_sweep_experiment --method pltreeshap_fasttreeshap --result_location {DRIVE_RESULT_PATH}"


[py38-setup] cd /content/treebranchmarks

[py38-setup] Installing micromamba under /content/micromamba ...

[py38-setup] Creating env 'py38' with Python 3.8 ...
[+] 0.0s
[+] 0.1s
conda-forge/linux-64   1%
conda-forge/noarch    ⣾  [+] 0.2s
conda-forge/linux-64  11%
conda-forge/noarch    13%[+] 0.3s
conda-forge/linux-64  20%
conda-forge/noarch    30%[+] 0.4s
conda-forge/linux-64  28%
conda-forge/noarch    48%[+] 0.5s
conda-forge/linux-64  37%
conda-forge/noarch    66%[+] 0.6s
conda-forge/linux-64  46%
conda-forge/noarch    84%[+] 0.7s
conda-forge/linux-64  54%
conda-forge/noarch    93%conda-forge/noarch                                
[+] 0.8s
conda-forge/linux-64  54%[+] 0.9s
conda-forge/linux-64  61%[+] 1.0s
conda-forge/linux-64  76%[+] 1.1s
conda-forge/linux-64  91%[+] 1.2s
conda-forge/linux-64  99%[+] 1.3s
conda-forge/linux-64  99%[+] 1.4s
conda-forge/linux-64  99%conda-forge/linux-64                              


Transaction

  Prefix: /content/micromamba/envs/py38

  Updating sp